# Age, gender, occupation, zip

`u.user` has one row per user: age, gender, occupation, and zip code. There are 670 men and 273 women, and $670 + 273 = 943$.

Ages run from 7 to 73. The sum of the ages is 32111.

$$
\dfrac{32111}{943} = 34 + \dfrac{49}{943}.
$$

User 30 is the 7-year-old: male, occupation student, zip 55436. User 481 is the 73-year-old: male, retired, zip 37771. One child and one retired user are not a reason to treat age as the main predictor of a star rating. The release kept these rows because the demographic fields were filled, and because each person had at least 20 ratings.

`u.occupation` lists 21 names. The largest group is `student`, 196 people. Then `other` 105, `educator` 95, `administrator` 79, and `engineer` 67. The name `other` is an occupation code in this file, and so is `none`. They are not the same code.

The zip code is stored as text. User 1's is 85711 and user 405's is 10019. This path does not turn those strings into distances. Two people in the same zip can disagree about *Star Wars*, and the rating matrix already records that disagreement when both of them rated it.


In [2]:
# Locate the MovieLens tables beside this notebook, or under the course root.
from pathlib import Path

def data_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "15-Movie Recommendation" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "u.info").exists():
            return direct
    raise FileNotFoundError(name)

def read_lines(name):
    return data_path(name).read_text(encoding="latin-1").splitlines()

def read_ratings(name):
    rows = []
    for line in read_lines(name):
        if not line.strip():
            continue
        user, item, rating, stamp = line.split("\t")
        rows.append((int(user), int(item), int(rating), int(stamp)))
    return rows

ratings = read_ratings("u.data")
print(len(ratings), ratings[0])


100000 (196, 242, 3, 881250949)


In [3]:
# Demographics: counts, the age fraction, and the two edge rows.
from fractions import Fraction
from collections import Counter
users = [line.split("|") for line in read_lines("u.user")]
genders = Counter(row[2] for row in users)
jobs = Counter(row[3] for row in users)
ages = [int(row[1]) for row in users]
print(genders, min(ages), max(ages), Fraction(sum(ages), len(ages)))
print(jobs.most_common(5))
by_id = {row[0]: row for row in users}
print(by_id["30"], by_id["481"])
assert genders == {"M": 670, "F": 273}
assert 670 + 273 == 943
assert (min(ages), max(ages), sum(ages)) == (7, 73, 32111)
assert Fraction(sum(ages), 943) == 34 + Fraction(49, 943)
assert jobs["student"] == 196
assert [name for name, _ in jobs.most_common(5)] == [
    "student", "other", "educator", "administrator", "engineer",
]
assert by_id["30"] == ["30", "7", "M", "student", "55436"]
assert by_id["481"] == ["481", "73", "M", "retired", "37771"]
assert "none" in jobs and "other" in jobs
assert len(read_lines("u.occupation")) == 21


Counter({'M': 670, 'F': 273}) 7 73 32111/943
[('student', 196), ('other', 105), ('educator', 95), ('administrator', 79), ('engineer', 67)]
['30', '7', 'M', 'student', '55436'] ['481', '73', 'M', 'retired', '37771']


## Pitfall

There are 196 students, and there is a user whose id is 196. User 196 is a writer, age 49. The occupation tally and the user id are stored in different files and do not name each other.

## Summary

670 men and 273 women. The mean age is $34 + 49/943$, with ages from 7 to 73. Twenty-one occupation names, 196 of them students. The zip code stays a string.
